
# AIE-CASE — Color-Invariant Saree Design Recognition

**Goal:** retrieve the same saree design despite changes in color palette, and verify whether two images depict the same design.

This notebook is intentionally compact enough for the Kaggle free GPU tier and is structured as an end-to-end, reproducible metric-learning pipeline.

### Core idea
1. Convert every image to **grayscale** before the encoder, removing direct hue information.
2. Use a compact **ImageNet-pretrained ResNet-18** encoder followed by a 128-D normalized embedding head.
3. Train with **supervised contrastive loss**, so images with the same `design_id` become close and different designs separate.
4. Use **cosine similarity** for gallery retrieval and verification.
5. Keep test designs unseen during training, while splitting each held-out design into gallery/query views so the evaluation measures the actual retrieval task.

> Replace the dataset paths/metadata mapping in the configuration cell with the supplied DeepLure corpus and the permitted public dataset. Do not redistribute the proprietary DeepLure corpus.



## 500-character approach note

I use a pretrained ResNet-18 as a compact metric-learning encoder, but convert every input to grayscale so the embedding cannot directly rely on hue. Strong geometric/contrast augmentation plus supervised contrastive learning makes same-design images close while separating different motifs. A cosine-similarity gallery performs identification; the validation split selects the verification threshold without touching test data.

**Length:** 428 characters.


In [ ]:

# 1. Configuration
import os, random, time, math, json, copy, re
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Point this at your local/Kaggle copy of the allowed data.
DATA_ROOT = Path("/kaggle/input")
METADATA_CSV = None  # e.g. "/kaggle/input/deeplure-metadata/metadata.csv"

IMAGE_SIZE = 224
EMBED_DIM = 128
BATCH_SIZE = 32
EPOCHS = 12
LR = 2e-4
WEIGHT_DECAY = 1e-4
TEMPERATURE = 0.07
NUM_WORKERS = 2
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"

print("device:", DEVICE)



## 2. Dataset contract

The cleanest reproducible format is a CSV with:

- `image_path`: path to the image
- `design_id`: the underlying design/motif identity
- `colorway`: optional colorway identifier

If the supplied corpus already uses one folder per design, the helper below can build a starter CSV automatically. **Inspect and correct the labels before training**; this is important because the assessment explicitly permits cleaning/relabeling and asks you to document it.


In [ ]:

# Build a starter metadata file when images are stored under design folders.
# Example:
#   corpus/
#     design_001/a.jpg
#     design_001/b.jpg
#     design_002/a.jpg

from PIL import Image

EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def build_metadata_from_design_folders(root):
    root = Path(root)
    rows = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXTS:
            rel = p.relative_to(root)
            if len(rel.parts) < 2:
                continue
            design_id = rel.parts[0]
            rows.append({"image_path": str(p), "design_id": str(design_id)})
    df = pd.DataFrame(rows)
    if len(df):
        df = df.sort_values(["design_id", "image_path"]).reset_index(drop=True)
    return df

if METADATA_CSV:
    df = pd.read_csv(METADATA_CSV)
else:
    # Edit this path if your corpus is not directly under DATA_ROOT.
    candidate = DATA_ROOT / "saree"
    df = build_metadata_from_design_folders(candidate)

assert len(df) > 0, (
    "No images found. Set METADATA_CSV to a CSV containing image_path/design_id "
    "or change candidate to the dataset image root."
)
assert {"image_path", "design_id"}.issubset(df.columns)

df["image_path"] = df["image_path"].map(str)
print(df.head())
print("images:", len(df), "designs:", df.design_id.nunique())
print(df.design_id.value_counts().describe())


In [ ]:

# 3. Leakage-resistant split
#
# Training designs are disjoint from validation/test designs.
# For validation/test designs, one or more images are reserved as queries and
# at least one image remains in the gallery. If colorway labels exist, we prefer
# holding out an entire colorway for the query side.

def split_by_design(df, seed=42, train_frac=.70, val_frac=.15):
    rng = np.random.default_rng(seed)
    designs = np.array(sorted(df.design_id.unique()))
    rng.shuffle(designs)
    n = len(designs)
    n_train = max(1, int(n * train_frac))
    n_val = max(1, int(n * val_frac)) if n >= 3 else 1
    train_ids = set(designs[:n_train])
    val_ids = set(designs[n_train:n_train+n_val])
    test_ids = set(designs[n_train+n_val:])
    if not test_ids:  # tiny smoke-test datasets
        test_ids = set(list(val_ids)[-1:])
        val_ids = val_ids - test_ids
    return train_ids, val_ids, test_ids

train_ids, val_ids, test_ids = split_by_design(df, SEED)

train_df = df[df.design_id.isin(train_ids)].reset_index(drop=True)
val_all = df[df.design_id.isin(val_ids)].reset_index(drop=True)
test_all = df[df.design_id.isin(test_ids)].reset_index(drop=True)

print({k: len(v) for k,v in {
    "train_images": train_df,
    "val_images": val_all,
    "test_images": test_all}.items()})
print("train/val/test designs:", len(train_ids), len(val_ids), len(test_ids))


In [ ]:

# Deterministically create gallery/query partitions for a held-out set.
# If colorway exists, try to put one complete colorway into the query side.
def make_gallery_query(part, seed=42):
    rng = np.random.default_rng(seed)
    gallery_rows, query_rows = [], []

    has_colorway = "colorway" in part.columns and part["colorway"].notna().any()

    for design_id, g in part.groupby("design_id"):
        g = g.sample(frac=1, random_state=int(rng.integers(0, 1_000_000))).reset_index(drop=True)

        if has_colorway:
            colors = [c for c in g.colorway.dropna().unique()]
            if len(colors) >= 2:
                q_color = colors[int(rng.integers(0, len(colors)))]
                q = g[g.colorway == q_color]
                gal = g[g.colorway != q_color]
                if len(q) and len(gal):
                    query_rows.append(q)
                    gallery_rows.append(gal)
                    continue

        # Fallback: image-level split within design.
        if len(g) >= 2:
            n_q = max(1, min(len(g)//3, len(g)-1))
            query_rows.append(g.iloc[:n_q])
            gallery_rows.append(g.iloc[n_q:])
        else:
            # A single image cannot form a valid retrieval pair.
            gallery_rows.append(g)

    gallery = pd.concat(gallery_rows, ignore_index=True) if gallery_rows else part.iloc[:0].copy()
    query = pd.concat(query_rows, ignore_index=True) if query_rows else part.iloc[:0].copy()
    return gallery, query

val_gallery, val_query = make_gallery_query(val_all, SEED)
test_gallery, test_query = make_gallery_query(test_all, SEED+1)

print("validation gallery/query:", len(val_gallery), len(val_query))
print("test gallery/query:", len(test_gallery), len(test_query))



## 4. Color-invariant preprocessing

The strongest domain-specific choice is to remove hue from the model input rather than hoping augmentation alone will teach invariance. We retain luminance/structure and use grayscale-compatible geometric and contrast augmentations. Random resized crops are intentionally moderate because textile motifs can be spatially repetitive and cropping too aggressively can destroy the identifying pattern.


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.70, 1.0), ratio=(0.80, 1.25)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.RandomRotation(8)], p=0.35),
    transforms.Grayscale(num_output_channels=3),
    transforms.ColorJitter(brightness=0.25, contrast=0.30, saturation=0.0, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize([0.485]*3, [0.229]*3),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485]*3, [0.229]*3),
])

class SareeDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        labels = sorted(self.frame.design_id.unique())
        self.label_map = {x:i for i,x in enumerate(labels)}
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        img = Image.open(row.image_path).convert("RGB")
        x = self.transform(img)
        y = self.label_map[row.design_id]
        return x, y

train_ds = SareeDataset(train_df, train_tfms)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)



## 5. Model

A compact ResNet-18 is used to keep the solution practical on the free GPU tier. The final classifier is removed and replaced with a small projection head. Embeddings are L2-normalized, so cosine similarity is simply a dot product.


In [ ]:

class MetricNet(nn.Module):
    def __init__(self, emb_dim=128):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        self.backbone = models.resnet18(weights=weights)
        in_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.15),
            nn.Linear(256, emb_dim),
        )

    def forward(self, x):
        z = self.head(self.backbone(x))
        return F.normalize(z, dim=1)

model = MetricNet(EMBED_DIM).to(DEVICE)
print("parameters:", sum(p.numel() for p in model.parameters()))



## 6. Supervised contrastive loss

Every batch is constructed from design labels. Same-design samples are positives; different designs are negatives. This directly optimizes the geometry needed by retrieval instead of training a closed-set classifier.


In [ ]:

def supervised_contrastive_loss(z, labels, temperature=0.07):
    z = F.normalize(z, dim=1)
    sim = torch.matmul(z, z.T) / temperature

    n = z.size(0)
    self_mask = torch.eye(n, device=z.device, dtype=torch.bool)
    sim = sim.masked_fill(self_mask, -1e9)

    pos = labels.unsqueeze(0).eq(labels.unsqueeze(1))
    pos = pos & (~self_mask)

    log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pos_count = pos.sum(1).clamp_min(1)
    mean_log_prob_pos = (log_prob * pos).sum(1) / pos_count

    valid = pos.sum(1) > 0
    if not valid.any():
        return sim.new_tensor(0.0)
    return -mean_log_prob_pos[valid].mean()


In [ ]:

# 7. Train
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

history = []
best_loss = float("inf")
best_state = None

for epoch in range(1, EPOCHS+1):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
            z = model(x)
            loss = supervised_contrastive_loss(z, y, TEMPERATURE)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item()

    scheduler.step()
    epoch_loss = running / max(1, len(train_loader))
    history.append(epoch_loss)
    print(f"epoch {epoch:02d} | loss {epoch_loss:.4f} | lr {scheduler.get_last_lr()[0]:.2e}")

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        best_state = copy.deepcopy(model.state_dict())

if best_state is not None:
    model.load_state_dict(best_state)



## 8. Embedding extraction and retrieval metrics


In [ ]:

@torch.no_grad()
def embed_frame(model, frame, batch_size=64):
    ds = SareeDataset(frame, eval_tfms)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    model.eval()
    zs = []
    for x, _ in dl:
        zs.append(model(x.to(DEVICE, non_blocking=True)).cpu())
    return torch.cat(zs, dim=0).numpy() if zs else np.empty((0, EMBED_DIM), dtype=np.float32)

def retrieval_metrics(gallery_df, query_df, gallery_z, query_z, ks=(1,5,10)):
    sims = query_z @ gallery_z.T
    gallery_ids = gallery_df.design_id.to_numpy()
    query_ids = query_df.design_id.to_numpy()

    order = np.argsort(-sims, axis=1)
    out = {}
    for k in ks:
        k2 = min(k, len(gallery_df))
        hits = [(gallery_ids[order[i,:k2]] == query_ids[i]).any()
                for i in range(len(query_df))]
        out[f"Recall@{k}"] = float(np.mean(hits)) if hits else float("nan")

    rr = []
    for i in range(len(query_df)):
        ranked_ids = gallery_ids[order[i]]
        pos = np.flatnonzero(ranked_ids == query_ids[i])
        rr.append(1.0/(pos[0]+1) if len(pos) else 0.0)
    out["MRR"] = float(np.mean(rr)) if rr else float("nan")
    return out

val_gz = embed_frame(model, val_gallery)
val_qz = embed_frame(model, val_query)
test_gz = embed_frame(model, test_gallery)
test_qz = embed_frame(model, test_query)

print("Validation:", retrieval_metrics(val_gallery, val_query, val_gz, val_qz))
print("Test:", retrieval_metrics(test_gallery, test_query, test_gz, test_qz))



## 9. Verification

For verification, the score is cosine similarity. The threshold is selected **only on validation pairs**, then frozen for test evaluation. This avoids tuning the test set.

We report ROC-AUC and TPR at a low false-positive operating point. The latter is useful when a false match is more costly than a missed match.


In [ ]:

from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

def make_pairs(frame, embeddings, seed=42, max_pairs=10000):
    rng = np.random.default_rng(seed)
    ids = frame.design_id.to_numpy()
    n = len(frame)
    pairs = []

    # Positive pairs
    by_design = {}
    for i, d in enumerate(ids):
        by_design.setdefault(d, []).append(i)
    for d, idxs in by_design.items():
        if len(idxs) >= 2:
            for a in range(len(idxs)):
                for b in range(a+1, len(idxs)):
                    pairs.append((idxs[a], idxs[b], 1))

    # Negative pairs
    neg_target = min(max_pairs, max(1, len(pairs)))
    design_list = list(by_design)
    while len([p for p in pairs if p[2] == 0]) < neg_target and len(design_list) >= 2:
        d1, d2 = rng.choice(design_list, 2, replace=False)
        i = int(rng.choice(by_design[d1])); j = int(rng.choice(by_design[d2]))
        pairs.append((i, j, 0))

    rng.shuffle(pairs)
    pairs = pairs[:max_pairs]
    scores = np.array([float(embeddings[i] @ embeddings[j]) for i,j,_ in pairs])
    y = np.array([label for _,_,label in pairs])
    return y, scores

def best_threshold(y, scores):
    thresholds = np.unique(scores)
    best_t, best_acc = 0.0, -1
    for t in thresholds:
        pred = (scores >= t).astype(int)
        acc = accuracy_score(y, pred)
        if acc > best_acc:
            best_acc, best_t = acc, float(t)
    return best_t, best_acc

val_y, val_s = make_pairs(val_all, embed_frame(model, val_all), seed=SEED)
test_y, test_s = make_pairs(test_all, embed_frame(model, test_all), seed=SEED+1)

threshold, val_acc = best_threshold(val_y, val_s)
test_auc = roc_auc_score(test_y, test_s)

fpr, tpr, thresholds = roc_curve(test_y, test_s)
target_fpr = 0.01
idx = np.where(fpr <= target_fpr)[0]
tpr_at_1pct = float(tpr[idx[-1]]) if len(idx) else float("nan")

print(f"validation threshold: {threshold:.4f}")
print(f"validation accuracy: {val_acc:.4f}")
print(f"test ROC-AUC: {test_auc:.4f}")
print(f"test TPR at FPR <= 1%: {tpr_at_1pct:.4f}")



## 10. Explicit color-invariance check

If `colorway` labels exist, this section measures **same-design / different-colorway** pairs separately. This is the most direct diagnostic for the central requirement of the assessment.

Also measure **different-design / same-colorway** negatives when such labels exist. The desired behavior is high similarity for the first group and low similarity for the second.


In [ ]:

def colorway_stress_test(frame, embeddings):
    if "colorway" not in frame.columns:
        print("No colorway column; skip explicit colorway stress test.")
        return None

    index = frame.reset_index(drop=True)
    groups = {}
    for i, r in index.iterrows():
        groups.setdefault((r.design_id, r.colorway), []).append(i)

    pos, neg = [], []
    keys = list(groups)
    for (d1,c1), idx1 in groups.items():
        for (d2,c2), idx2 in groups.items():
            if (d1,c1) >= (d2,c2):
                continue
            if d1 == d2:
                pos.append(float(embeddings[idx1[0]] @ embeddings[idx2[0]]))
            elif c1 == c2:
                neg.append(float(embeddings[idx1[0]] @ embeddings[idx2[0]]))

    out = {
        "same_design_different_colorway_mean": float(np.mean(pos)) if pos else None,
        "different_design_same_colorway_mean": float(np.mean(neg)) if neg else None,
        "positive_pairs": len(pos),
        "negative_pairs": len(neg),
    }
    return out

print(colorway_stress_test(test_all, embed_frame(model, test_all)))



## 11. Inference: identify and verify

For deployment, encode the gallery once and cache its embeddings. Each query then requires one forward pass plus a matrix multiplication against the gallery embeddings.


In [ ]:

@torch.no_grad()
def encode_one(model, image_path):
    img = Image.open(image_path).convert("RGB")
    x = eval_tfms(img).unsqueeze(0).to(DEVICE)
    return model(x).cpu().numpy()[0]

def identify(query_path, gallery_df, gallery_embeddings, top_k=5):
    q = encode_one(model, query_path)
    scores = gallery_embeddings @ q
    idx = np.argsort(-scores)[:top_k]
    result = gallery_df.iloc[idx][["image_path", "design_id"]].copy()
    result["cosine_similarity"] = scores[idx]
    return result.reset_index(drop=True)

def verify(image_a, image_b, threshold):
    za, zb = encode_one(model, image_a), encode_one(model, image_b)
    score = float(za @ zb)
    return {"same_design": score >= threshold, "cosine_similarity": score}

# Example:
# print(identify("/path/query.jpg", test_gallery, test_gz, top_k=5))
# print(verify("/path/a.jpg", "/path/b.jpg", threshold))



## 12. Efficiency report

The following cell records parameter count and a simple inference-latency measurement. For a final submission, run it on the actual Kaggle hardware and report the measured values rather than estimating them.


In [ ]:

def parameter_count(m):
    return sum(p.numel() for p in m.parameters())

@torch.no_grad()
def benchmark(model, n=100):
    model.eval()
    x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    for _ in range(10):
        _ = model(x)
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        _ = model(x)
    if DEVICE == "cuda": torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) * 1000 / n
    return ms

print("parameters:", parameter_count(model))
print("embedding dimension:", EMBED_DIM)
print("batch-1 inference ms:", benchmark(model))



## 13. What to report in the final submission

Do not invent metrics. After the notebook has run, copy the actual values into a short results table:

| Metric | Validation | Test |
|---|---:|---:|
| Recall@1 | run | run |
| Recall@5 | run | run |
| MRR | run | run |
| Verification ROC-AUC | run | run |
| TPR @ FPR ≤ 1% | — | run |

Also report:
- number of images and unique designs after cleaning;
- how colorways were identified;
- exact gallery/query construction;
- pretrained checkpoint and torchvision version;
- epochs, batch size, learning rate, seed;
- parameter count, embedding size and measured latency;
- any external/public data used;
- confirmation that the proprietary DeepLure corpus was not redistributed.

### Defensible design decisions
- **Why grayscale?** It removes direct color information at the input, matching the problem definition.
- **Why metric learning?** The output is similarity/ranking, not a fixed closed-set class prediction.
- **Why design-disjoint training?** It prevents the encoder from memorizing test design identities.
- **Why gallery/query within held-out designs?** Retrieval needs the same design represented on both sides, while colorway separation tests invariance.
- **Why validation-selected threshold?** It prevents test-set threshold tuning.
- **Why ResNet-18?** It is small enough for the free GPU tier and easy to reproduce/defend.

### Important caveat
If the metadata does not contain trustworthy `design_id`/`colorway` labels, fix the metadata first. The quality of a color-invariance metric-learning system is bounded by the correctness of its positive and negative pairs.
